# CI/CD: Automated Testing and Deployment

## The Core Question

**How do you automatically test your code on every commit?**

You push a change to your 1000-line CFD script. It *seems* fine. But did you break a test? Did you introduce a lint error? Did you break the build? You won't know until a colleague runs it and it fails — or worse, until a simulation produces wrong results.

CI/CD runs your tests, linters, and build **on every push**, automatically. Red = broken, green = safe.

## Why this library?

GitHub Actions is a CI/CD platform built into GitHub. You write a single YAML file (`.github/workflows/ci.yml`) that describes a **pipeline**: install dependencies, run the linter, run the type checker, run the tests. Every push to `main` triggers it. Unlike manual testing, it's **automatic, consistent, and recorded** — you always know the last commit that was green.

## The Workflow

```
  git push to main
        |
        v
  GitHub Actions triggers ci.yml
        |
        +--> install deps (pip install -r requirements.txt)
        +--> ruff check        (lint)
        +--> mypy              (type check)
        +--> pytest            (tests)
        |
        v
  green = safe to merge   |   red = broken, fix before merging
```


In [ ]:
# The tools the pipeline will run. In a real project these live in
# requirements.txt (or are managed by Poetry, as in the previous notebook).
import subprocess, sys
for tool in ['pytest', 'ruff', 'mypy']:
    out = subprocess.run([sys.executable, '-m', tool, '--version'],
                         capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr.strip())

## 1. The CI Workflow File

A GitHub Actions workflow is a YAML file in `.github/workflows/`. It has three parts:

- **`on`** — *when* to run (here: every push to `main`, and every pull request).
- **`jobs`** — *what* to run. Each job runs on a fresh virtual machine.
- **`steps`** — the ordered commands inside a job.

Let's write a minimal but complete pipeline for an engineering project.


In [ ]:
# .github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Check out the code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install pytest ruff mypy

      - name: Lint with Ruff
        run: ruff check .

      - name: Type-check with Mypy
        run: mypy .

      - name: Run tests with Pytest
        run: pytest -v

### Reading the YAML

- **`on: push / pull_request`** — the pipeline runs on every push to `main` *and* on every PR targeting `main`. That means you get feedback *before* merging, not after.
- **`runs-on: ubuntu-latest`** — each run gets a clean Linux VM. No "works on my machine" — it's a fresh environment every time.
- **`actions/checkout@pinned`** — checks out your code. Pinning the version (`@v4`) keeps the action itself reproducible.
- **`actions/setup-python`** — installs Python 3.12.
- **The three quality gates** — `ruff check` (lint), `mypy` (types), `pytest` (tests). If *any* step fails, the whole run is **red** and the PR can't be merged (if you enforce it).

**Engineering takeaway:** this is the "safety net" for your simulation code. Your 1000-line CFD script gets linted, type-checked, and tested on *every commit*. No more "I broke it and didn't notice" — the red badge tells you within minutes.


### Validate the YAML

A workflow file is just YAML — you can parse it locally to catch syntax errors before pushing. Let's confirm the file above is well-formed.


In [ ]:
import yaml, textwrap

ci_yaml = textwrap.dedent('''
    name: CI

    on:
      push:
        branches: [main]
      pull_request:
        branches: [main]

    jobs:
      test:
        runs-on: ubuntu-latest
        steps:
          - name: Check out the code
            uses: actions/checkout@v4
          - name: Set up Python
            uses: actions/setup-python@v5
            with:
              python-version: "3.12"
          - name: Install dependencies
            run: |
              python -m pip install --upgrade pip
              pip install -r requirements.txt
              pip install pytest ruff mypy
          - name: Lint with Ruff
            run: ruff check .
          - name: Type-check with Mypy
            run: mypy .
          - name: Run tests with Pytest
            run: pytest -v
''')

data = yaml.safe_load(ci_yaml)
print('Valid YAML. Top-level keys:', list(data.keys()))
print('Jobs:', list(data['jobs'].keys()))
print('Steps in "test" job:', len(data['jobs']['test']['steps']))
for step in data['jobs']['test']['steps']:
    print('  -', step['name'])

### Reading the parsed structure

The YAML parses cleanly into a dict: one job (`test`) with six ordered steps. This is exactly what GitHub Actions will execute. Parsing it locally means you catch a missing colon or bad indentation *before* you push — not after a failed run.

**Engineering takeaway:** treat your workflow file like code. Lint it, review it in a PR, and keep it version-controlled. It's part of your engineering infrastructure.


## 2. What a Run Looks Like

When you push, GitHub Actions runs the pipeline and shows each step. A **green** run means all gates passed:

```
✓ Check out the code
✓ Set up Python
✓ Install dependencies
✓ Lint with Ruff
✓ Type-check with Mypy
✓ Run tests with Pytest
  8 passed in 0.02s
```

A **red** run stops at the first failing step and shows you exactly what broke:

```
✓ Check out the code
✓ Set up Python
✓ Install dependencies
✗ Lint with Ruff
  F401 [*] `numpy` imported but unused
  Found 1 error.
```

The red run tells you *which* gate failed and *why* — so you fix it before it reaches `main`.


## 3. Why This Matters for Engineering Code

Manual testing has a fatal flaw: **you skip it when you're in a hurry.** CI/CD removes the human from the loop. The pipeline runs whether you remember or not.

For a simulation engineer, this means:

- **Physics tests run on every change.** A sign flip or unit error in your solver is caught before it propagates to a 1000-hour run.
- **The environment is consistent.** Every run uses the same Python version and dependencies — no drift.
- **There's a record.** You can always find the last green commit. When a result looks wrong, you know which code produced it.

**Engineering takeaway:** CI/CD is the difference between "I *think* this works" and "this *is verified* to work." It's the cheapest insurance you can buy for expensive simulations.


## 4. Exercises

1. **Add a build step.** Extend the workflow with a step that builds your package (`python -m build`) and uploads the wheel as an artifact. How would you download it in a later job?

2. **Matrix testing.** Run the tests on multiple Python versions (3.10, 3.11, 3.12) using a `strategy.matrix`. How does the YAML change?

3. **Fail a gate on purpose.** Push a commit with an unused import. Watch the `ruff check` step go red. Read the failure output — what does it tell you?

4. **Protect the branch.** In your repo settings, require a green CI status before merging to `main`. What does that enforce, and why is it important for a shared simulation codebase?
